# EXP-2026-007 training series E: train E0s or E3m (GPU)

Card `experiments/registry/EXP-2026-007.md` (approved 2026-11-25, decisions 1 to 9, revised after Codex's review). Run
this notebook only after the first session (the evaluation notebook with `SMOKE_FIRST = True` and `MODEL = "E0"`) has
passed: decision 9.

- **E0s:** r2's recipe with train seed 2027, RHUH's 24 accepted training patients. It measures training noise.
- **E3m:** r2's recipe with train seed 2026, plus LUMIERE run r2's 25 accepted training-role pairs as image-only rows:
  empty prompts, and a mask row of pure noise at timestep 999 with loss weight 0.
- **Start:** r2's step-0 weights. The author checkpoint must have the SHA-256 recorded by Stage 5 run r2; it is widened
  from 4 to 11 inputs with zero columns, and the optimizer starts fresh. Nothing continues from E0.
- **Checks before training:** cohort membership equals the accepted training sets exactly (no validation, test or
  development case), and the cache's lineage (members, split and policy hashes, export hashes, settings) is checked
  again right before training. The run stops on any mismatch.
- **Output:** the step-1,000 weights and a write-once record, uploaded through the Drive API to
  `MyDrive/RHUH-GBM/synthesis/EXP-2026-007/train/<RUN_TAG>` and confirmed by MD5. The record names patients: keep that
  folder private. Only the step-1,000 weights are evaluated.

Runtime: a G4 GPU. About 1 to 1.5 hours. Keep the tab open until the last cell finishes.

**Clear all outputs before saving, sharing or committing this notebook.**

In [ ]:
REPO_URL = "https://github.com/AIVIETNAM-AIO-Tuan/MedSegFactory.git"  # your fork only
BRANCH = "lumiere-onboarding"
WORK = "/content/MedSegFactory-e-train"
DRIVE_RHUH = "/content/drive/MyDrive/RHUH-GBM"
SD_REPO = "stable-diffusion-v1-5/stable-diffusion-v1-5"
MSF_REPO, MSF_FILE = "JohnWeck/StableDiffusion", "checkpoint-300.pth"
RUN = "E0s"  # "E0s" or "E3m"
RUN_TAG = f"{RUN}_r1"  # a new tag for every attempt; a run folder that holds files is refused
DATA_ON_DRIVE = ("RHUH-GBM", "synthesis", "EXP-2026-001")  # from MyDrive; the API follows the shortcut
OUT_BASE, OUT_RUN = ("RHUH-GBM", "synthesis"), ("EXP-2026-007", "train", RUN_TAG)
LUMIERE_RUN = ("LUMIERE_derived", "EXP-2026-005", "r2")  # read only, used by E3m

## 1. Code from the fork, packages, Drive API, run folders

In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")  # before any CUDA work: repeatable cuBLAS
from google.colab import drive

drive.mount("/content/drive")  # RHUH exports are read through the mount; every write goes through the Drive API
if not Path(WORK).is_dir():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORK], check=True)
else:
    current = subprocess.check_output(["git", "-C", WORK, "branch", "--show-current"], text=True).strip()
    assert current == BRANCH, "This checkout belongs to another task; choose a different WORK"
subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
link = Path(WORK, "data", "RHUH-GBM")
link.parent.mkdir(exist_ok=True)
if not link.exists():
    link.symlink_to(DRIVE_RHUH, target_is_directory=True)
os.chdir(WORK)
sys.path.insert(0, WORK)

from rhuh_synthesis import colab_packages  # the one package list; it imports nothing (a module, so no name clash)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", *colab_packages.DATA, *colab_packages.MODEL], check=True)
commit = subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip()
print(commit)

from google.colab import auth
from googleapiclient.discovery import build

from rhuh_synthesis.drive_api import DriveRun

auth.authenticate_user()  # Colab asks once for permission to use the Drive API with this account
api = build("drive", "v3", cache_discovery=False)

assert RUN in ("E0s", "E3m"), "RUN must be E0s or E3m"
LOCAL_RUN = Path(f"/content/e_train_{RUN_TAG}")
assert not LOCAL_RUN.exists(), f"{LOCAL_RUN} exists: choose a new RUN_TAG or a fresh runtime"
settings = json.loads(Path("configs/experiments/EXP-2026-007.json").read_text(encoding="utf-8"))
r2_run = DriveRun.open(api, DATA_ON_DRIVE, ("stage5_runs", "r2"), "/content/e_train_r2_record",
                       expect_in_base=("split", "export_v1"))  # read only: r2's record
# Base by path (shortcuts and folders shared with this account are followed), run folder below it
lumiere = (DriveRun.open(api, LUMIERE_RUN[:-1], LUMIERE_RUN[-1:], "/content/e_train_lumiere_r2", create=False)
           if RUN == "E3m" else None)
run_drive = DriveRun.open(api, OUT_BASE, OUT_RUN, LOCAL_RUN, expect_in_base=("EXP-2026-001",), create=True)
access = run_drive.preflight()  # a real write, update, read and removal on Drive before any GPU time
print("free GiB on Drive:", None if access["free_gib"] is None else round(access["free_gib"], 1))
assert access["free_gib"] is None or access["free_gib"] > 6, "The run needs about 6 GiB free on this account's Drive"
STARTED = time.perf_counter()


def say(message):
    print(f"[{time.perf_counter() - STARTED:6.0f} s] {message}", flush=True)

## 2. r2's starting weights, checked, and the frozen encoders

In [ ]:
import numpy as np
import torch
from diffusers import AutoencoderKL, DDPMScheduler
from huggingface_hub import hf_hub_download
from transformers import CLIPTextModel, CLIPTokenizer

from rhuh_synthesis.e_series import state_dict_sha256
from rhuh_synthesis.manifest import file_sha256
from rhuh_synthesis.model import expand_conv_in
from rhuh_synthesis.provenance import code_version, config_sha256, environment
from rhuh_synthesis.stage3 import adapter_equivalence, clip_encoder, load_reference_unet
from StableDiffusion.Our_UNet import UNet2DConditionModel

config = json.loads(Path("configs/experiments/EXP-2026-001.json").read_text(encoding="utf-8"))  # frozen, read only
S4, TRAIN = config["stage4_overfit"], config["stage5_pilot"]["training"]
r2 = json.loads(r2_run.fetch("record.json").read_text(encoding="utf-8"))  # by ID, checked by MD5
checkpoint = hf_hub_download(MSF_REPO, MSF_FILE)
checkpoint_sha = file_sha256(checkpoint)
assert checkpoint_sha == r2["checkpoint"]["sha256"], "The author checkpoint differs from r2's: stop and report"
unet4, load_report = load_reference_unet(UNet2DConditionModel, UNet2DConditionModel.load_config(SD_REPO, subfolder="unet"),
                                         checkpoint)
assert unet4 is not None, "Checkpoint keys do not match the vendored UNet: stop and report"
off = adapter_equivalence(unet4, latent_hw=(64, 64), cross_dim=768, device="cuda", tf32=False)
limits = S4["preflight_adapter_equivalence"]
assert off["relative_difference"] <= limits["relative_difference_max"] and off["copied_weights_identical"] and off["added_weights_zero"], \
    "Adapter preflight failed: stop and report"
unet = expand_conv_in(unet4, 11).to("cuda")
del unet4
torch.cuda.empty_cache()
step0_sha = state_dict_sha256(unet)
vae = AutoencoderKL.from_pretrained(SD_REPO, subfolder="vae").to("cuda").eval().requires_grad_(False)
tokenizer = CLIPTokenizer.from_pretrained(SD_REPO, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(SD_REPO, subfolder="text_encoder").to("cuda").eval().requires_grad_(False)
noise_scheduler = DDPMScheduler.from_pretrained(SD_REPO, subfolder="scheduler")
encode_text = clip_encoder(tokenizer, text_encoder, "cuda")
record = {
    "experiment": "EXP-2026-007", "run": RUN, "run_tag": RUN_TAG, "commit": commit, "code_version": code_version(WORK),
    "settings_sha256": config_sha256(settings), "exp001_config_sha256": config_sha256(config),
    "environment": environment(), "torch": torch.__version__, "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0), "train_seed": settings["runs"][RUN]["train_seed"],
    "start": {"author_checkpoint_sha256": checkpoint_sha, "equals_r2": True, "step0_state_sha256": step0_sha,
              "adapter_equivalence": off, "optimizer": "fresh AdamW"},
}
say(f"models ready; step-0 weights {step0_sha[:12]}")

## 3. The training cache, its members and its lineage

In [ ]:
from rhuh_synthesis.e_series import (attach_lineage, check_cache, check_members, file_sha256 as sha_of_file, make_lineage,
                                     rhuh_training_members, settings_of, sha256_of)
from rhuh_synthesis.lumiere_scaffold_check import accepted_training_pairs, verified_split
from rhuh_synthesis.pilot import local_copies, training_cache
from rhuh_synthesis.representation import crop_forward
from rhuh_synthesis.training import build_image_only_cache, image_only_pools, merge_caches
from rhuh_synthesis.working_grid import to_image_only_grid, to_working_grid

PRIVATE = Path("data/RHUH-GBM/synthesis/EXP-2026-001")
split = json.loads((PRIVATE / "split/split.json").read_text(encoding="utf-8"))
policy = json.loads((PRIVATE / "registration_policy/acceptance_policy_v1.json").read_text(encoding="utf-8"))
members = rhuh_training_members(split, policy)
assert len(members) == r2["data_summary"]["training_patients"], "RHUH's training set differs from r2's: stop and report"
paths = local_copies({p: next(PRIVATE.glob(f"export_v1/train*/{p}.npz")) for p in members}, "/content/e_train_exports")


def load_grid(patient):
    with np.load(paths[patient]) as data:
        arrays = {key: data[key] for key in data.files}
    crop, grid = to_working_grid(arrays)
    grid["post_support"] = crop_forward(arrays["post_support"].astype(np.uint8), crop).astype(bool)
    return grid


labelled, pools = training_cache(members, load_grid, vae, encode_text, config["text"],
                                 mask_codec=config["representation"]["mask_codec"], batch=16, device="cuda")
rhuh = {"split_sha256": sha256_of(split), "policy_sha256": sha256_of(policy),
        "exports": {p: sha_of_file(paths[p]) for p in members}}
lumiere_part, lumiere_accepted, image_only, lumiere_split = None, None, None, None
if RUN == "E3m":
    for name in ("run.json", "split.json", "registration_qc.csv", "summary.json"):
        lumiere.fetch(f"outputs/{name}")
    outputs = Path("/content/e_train_lumiere_r2/outputs")
    lumiere_split, lumiere_split_sha, _ = verified_split(outputs)
    files = lumiere.fetch_folder("outputs/export/train")  # training-role exports and sidecars only
    pairs, counts = accepted_training_pairs(outputs, lumiere_split, lumiere_split_sha)
    assert counts["accepted"] == settings["lumiere_source"]["expected_train_accepted"], "LUMIERE's accepted pairs differ from the card"
    for path, sidecar in pairs:
        with np.load(path) as data:
            arrays = {key: data[key] for key in data.files}
        _, grid = to_image_only_grid(arrays)
        one = build_image_only_cache({path.stem: grid}, vae, encode_text, cohort="LUMIERE", batch=16, device="cuda")
        if image_only is None:
            image_only = one
        else:
            image_only["patients"].update(one["patients"])
    pools.update(image_only_pools(image_only))
    lumiere_accepted = {path.stem for path, _ in pairs}
    lumiere_part = {"run": settings["lumiere_source"]["run"], "split_sha256": lumiere_split_sha,
                    "exports": {path.stem: sidecar["npz_sha256"] for path, sidecar in pairs}}
cache = merge_caches(labelled, image_only, cohort="RHUH")
del labelled
lineage = make_lineage(rhuh=rhuh, lumiere=lumiere_part,
                       settings=settings_of(config, sd_repo=SD_REPO, scaling_factor=vae.config.scaling_factor))
counts = check_members(lineage, rhuh_split=split, rhuh_policy=policy, lumiere_split=lumiere_split,
                       lumiere_accepted=lumiere_accepted)
attach_lineage(cache, lineage)
record.update(members=counts, lineage=lineage, lineage_sha256=cache["lineage_sha256"])
say(f"cache ready: {counts}, {sum(len(p['positive']) + len(p['negative']) for p in pools.values())} slices; "
    f"lineage {cache['lineage_sha256'][:12]}")

## 4. 1,000 steps of r2's recipe

In [ ]:
from rhuh_synthesis.training import SliceSampler, train

check_cache(cache, lineage)  # again, right before training
seed = settings["runs"][RUN]["train_seed"]
optimizer = torch.optim.AdamW(unet.parameters(), lr=config["training"]["learning_rate"],
                              weight_decay=config["training"]["weight_decay"])
sampler = SliceSampler(pools, seed=seed, positive_pool_probability=config["training"]["positive_pool_probability"])
rng, generator = np.random.default_rng(seed), torch.Generator().manual_seed(seed)
LOCAL_RUN.mkdir(parents=True, exist_ok=True)
log_path = LOCAL_RUN / "train_log.jsonl"
CHECKPOINTS = Path(f"/content/e_train_{RUN_TAG}_checkpoints")  # full resume checkpoints, local only
torch.cuda.reset_peak_memory_stats()
start = time.perf_counter()
step = train(unet, cache, sampler, noise_scheduler, optimizer, steps=TRAIN["steps"], accumulation=TRAIN["accumulation"],
             microbatch=TRAIN["microbatch_pairs"], rng=rng, generator=generator, log_path=log_path,
             checkpoint_dir=CHECKPOINTS, checkpoint_every=config["training"]["checkpoint_every"], start_step=0,
             precision=TRAIN["precision"], config_sha256=record["exp001_config_sha256"], cohort_stats=True)
assert step == settings["training"]["evaluated_checkpoint_step"] == TRAIN["steps"]
record["train_seconds"] = time.perf_counter() - start
record["peak_gib"] = torch.cuda.max_memory_allocated() / 2**30
say(f"trained to step {step} in {record['train_seconds']:.0f} s, peak {record['peak_gib']:.1f} GiB")

## 5. The step-1,000 weights and the write-once record

In [ ]:
from rhuh_synthesis.e_series import cohort_summary
from rhuh_synthesis.manifest import write_once_json

weights = LOCAL_RUN / f"unet_step{step:06d}.pt"
torch.save(unet.state_dict(), weights)
record["weights"] = {"file": weights.name, "sha256": file_sha256(weights), "step": step}
training_log = [json.loads(line) for line in log_path.read_text(encoding="utf-8").splitlines()]
losses = [line["loss"] for line in training_log]
record.update(training_log=training_log, cohorts=cohort_summary(training_log),
              sampler_counts=sampler.state_dict()["counts"], total_seconds=time.perf_counter() - STARTED)
path, digest = write_once_json(LOCAL_RUN / "record.json", record)
report = run_drive.sync()
say(f"record {digest[:12]} written; Drive upload: {report}")
shareable = {
    "commit": commit, "run": RUN, "train_seed": record["train_seed"], "gpu": record["gpu"],
    "author_checkpoint_sha256": checkpoint_sha[:12], "step0_state_sha256": step0_sha[:12],
    "weights_sha256": record["weights"]["sha256"][:12], "lineage_sha256": record["lineage_sha256"][:12],
    "members": record["members"], "cohorts": record["cohorts"],
    "train_loss": {"mean_first_50": float(np.mean(losses[:50])), "mean_at_500": float(np.mean(losses[450:500])),
                   "mean_last_50": float(np.mean(losses[-50:])), "all_finite": all(l["finite"] for l in training_log)},
    "train_seconds": round(record["train_seconds"]), "total_seconds": round(record["total_seconds"]),
    "peak_gib": round(record["peak_gib"], 1),
}

## 6. Confirm the upload and print the summary

In [ ]:
assert run_drive.confirm(), f"Not every file is confirmed on Drive. The originals are in {LOCAL_RUN}: keep this session open and report"
print("Every file confirmed by the Drive server: MyDrive/" + "/".join(OUT_BASE + OUT_RUN))
print("SUMMARY")
print(json.dumps(shareable, indent=1))

## What to send back

Copy everything printed after `SUMMARY`. It holds counts, losses and short hashes, no patient name.

If a cell stops, send the error message. The run stops on purpose when the author checkpoint, the cohort, a split, an
export or the cache's lineage does not match.